In [0]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import uuid

spark = SparkSession.builder.getOrCreate()

def ingest_to_bronze(source_path: str, table_name: str, batch_id: str):
    """
    Função genérica para ingestão na camada Bronze.
    
    [ADR - Decisão de Arquitetura]: 
    O desafio questiona o uso de Auto Loader em produção. 
    Sim, em um ambiente produtivo real, utilizaríamos o Databricks Auto Loader (cloudFiles) 
    para orquestrar a ingestão incremental nativamente, pois ele gerencia o estado dos arquivos 
    já processados de forma escalável e barata via RocksDB. Como aqui estamos lendo um dump 
    estático de arquivos para o case, simularemos o controle de lotes via parâmetro 'batch_id'.
    """
    print(f"Iniciando ingestão da tabela: bronze_{table_name}")
    
    # 1. Leitura do arquivo bruto inferindo schema (preservando campos inesperados)
    df_raw = spark.read.option("header", True).option("inferSchema", True).csv(source_path)
    
    # 2. Adição dos metadados de governança obrigatórios
    colunas_originais = df_raw.columns
    
    df_bronze = (
        df_raw
        # Correção Unity Catalog: Usando a coluna oculta de metadados
        .withColumn("arquivo_origem", F.col("_metadata.file_path"))
        .withColumn("data_ingestao", F.current_date())
        .withColumn("timestamp_ingestao", F.current_timestamp())
        .withColumn("batch_id", F.lit(batch_id))
        # Gera o hash concatenando as colunas originais para rastreabilidade de linha
        .withColumn("hash_linha", F.md5(F.concat_ws("||", *[F.col(c).cast("string") for c in colunas_originais])))
        .withColumn("schema_version", F.lit("1.0"))
    )
    
    # 3. Escrita incremental no formato Delta Lake
    # O mergeSchema=True garante que novas colunas no futuro não quebrem o pipeline
    (df_bronze.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(f"workspace.default.bronze_{table_name}")
    )
    
    print(f"Sucesso! Tabela workspace.default.bronze_{table_name} atualizada.")

# ==========================================
# Célula de Execução do Pipeline
# ==========================================
meu_batch_id = str(uuid.uuid4())
volume_path = "/Volumes/workspace/default/raw_data"

# Executa a ingestão para as nossas 3 fontes de dados simuladas
ingest_to_bronze(f"{volume_path}/clientes_cdc.csv", "clientes", meu_batch_id)
ingest_to_bronze(f"{volume_path}/contas_cdc.csv", "contas", meu_batch_id)
ingest_to_bronze(f"{volume_path}/cartoes_cdc.csv", "cartoes", meu_batch_id)